# S06-13C — Visualise the BGR Graph

Rebuilds the building model from **13A** and draws the adjacency graph on top of it using `Topology.Show` — the same approach used in HW01 and HW02.

| cell_type | name   | colour    |
|-----------|--------|-----------|
| 0         | ground | green     |
| 1         | column | gray      |
| 3         | office | lightblue |
| 4         | core   | orange    |

## 1. Setup

In [12]:
from topologicpy.Vertex import Vertex
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

SUPPORT_DIR = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework03\Supporting Files"
renderer = "vscode"

## 2. Rebuild model (13A steps 1–4)

Import OBJs, tag each cell, merge into one CellComplex, and transfer dictionaries.

In [13]:
column_objs = Topology.ByOBJPath(SUPPORT_DIR + r"\columns.obj", transposeAxes=False)
office_objs = Topology.ByOBJPath(SUPPORT_DIR + r"\offices.obj", transposeAxes=False)
core_objs   = Topology.ByOBJPath(SUPPORT_DIR + r"\core.obj",    transposeAxes=False)
ground_objs = Topology.ByOBJPath(SUPPORT_DIR + r"\ground.obj",  transposeAxes=False)

selectors  = []
all_cells  = []

for objs, ctype, cname, ccolor in [
    (ground_objs, 0, "ground", "green"),
    (column_objs, 1, "column", "gray"),
    (office_objs, 3, "office", "lightblue"),
    (core_objs,   4, "core",   "orange"),
]:
    faces = Helper.Flatten([Topology.Faces(o) for o in objs if Topology.IsInstance(o, "Topology")])
    cells = Topology.Cells(Topology.SelfMerge(Cluster.ByTopologies(faces)))
    for cell in cells:
        d = Dictionary.ByKeysValues(
            ["cell_type", "cell_name", "cell_color"],
            [ctype, cname, ccolor]
        )
        s = Topology.InternalVertex(cell)
        s = Topology.SetDictionary(s, d)
        selectors.append(s)
    all_cells += cells

model = Topology.SelfMerge(Cluster.ByTopologies(all_cells))
model = Topology.TransferDictionariesBySelectors(model, selectors, tranCells=True)
print(f"Model ready: {len(Topology.Cells(model))} cells, {len(selectors)} selectors")

Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: <module>
Model ready: 45 cells, 39 selectors


## 3. Build the graph

`Graph.ByTopology` converts the CellComplex into an adjacency graph — each cell becomes a node, each shared face becomes an edge.

In [14]:
graph    = Graph.ByTopology(model)
vertices = Graph.Vertices(graph)
edges    = Graph.Edges(graph)
print(f"Graph: {len(vertices)} nodes, {len(edges)} edges")

Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: _append_edge
DEBUG e (Edge): None
DEBUG - src: 2
DEBUG - dst: 3
DEBUG - gv1 coordinates: [75.25, 21.75, 1.5]
DEBUG - gv2 coordinates: [75.25, 21.75, 1.5]
Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: _append_edge
DEBUG e (Edge): None
DEBUG - src: 10
DEBUG - dst: 11
DEBUG - gv1 coordinates: [59.75, 0.25, 1.5]
DEBUG - gv2 coordinates: [59.75, 0.25, 1.5]
Edge.ByStartVertexEndVertex - Error: The distance between the input vertexA and vertexB parameters is less than the input tolerance. Returning None.
caller name: _append_edge
DEBUG e (Edge): None
DEBUG - src: 12
DEBUG - dst: 13
DEBUG - gv1 coordinates: [34.75, 0.25, 1.5]
DEBUG - gv2 coordinates: [34.75, 0.25, 1.5]
Edge.ByStartVertexEndVertex - Error: The distance be

## 4. Style vertices and edges

Same colours as the 13A model: green / gray / lightblue / orange.
Ground node is drawn larger to highlight its hub role.

In [15]:
# exact same colour strings used in 13A
COLOR_BY_TYPE = {0: "green", 1: "gray", 3: "lightblue", 4: "orange"}
SIZE_BY_TYPE  = {0: 20,      1: 8,      3: 12,          4: 14}

for v in vertices:
    d         = Topology.Dictionary(v)
    cell_type = Dictionary.ValueAtKey(d, "cell_type")
    d = Dictionary.SetValueAtKey(d, "color", COLOR_BY_TYPE.get(cell_type, "white"))
    d = Dictionary.SetValueAtKey(d, "size",  SIZE_BY_TYPE.get(cell_type, 10))
    Topology.SetDictionary(v, d)

for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [1, "white"])
    Topology.SetDictionary(e, d)

print("Vertices and edges styled.")

Vertices and edges styled.


## 5. Show graph overlaid on building model

In [16]:
Topology.Show(
    model, graph,
    faceColorKey="cell_color",
    faceOpacity=0.15,
    showEdges=True,
    showVertices=True,
    vertexSizeKey="size",
    vertexColorKey="color",
    edgeWidthKey="width",
    edgeColorKey="color",
    backgroundColor="black",
    width=900,
    height=700,
    renderer=renderer
)

## 6. Show graph alone (no geometry)

Removes the model to make adjacency structure easier to read.

In [17]:
Topology.Show(
    graph,
    showEdges=True,
    showVertices=True,
    vertexSizeKey="size",
    vertexColorKey="color",
    edgeWidthKey="width",
    edgeColorKey="color",
    backgroundColor="black",
    width=900,
    height=700,
    renderer=renderer
)

## 7. Export interactive Pyvis graph

Saves an HTML file you can open in a browser — nodes are draggable and hoverable.

## 8. Compare Prediction vs True visually

Following the pattern from **S06-15 section 16**.

Loads the graph from CSV, re-runs the BGR prediction, then shows two views:
- **True** — nodes coloured by `cell_type` (your assigned label)
- **Predicted** — same colours if the model agreed; nodes turn **red** where the graph-level label was wrong

`Graph.Reshape` flattens the 3D graph into a clean 2D spring layout for readability.

In [18]:
import pandas as pd
from pathlib import Path
from topologicpy.PyG import PyG
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology

CSV_DIR    = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework03\CSV")
MODEL_PATH = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML\example_dataset\dataset_graph_classification\bgr_model.pt")

MAPPING = {0: "Separation", 1: "Separation with Plinth",
           2: "Adherence",  3: "Adherence with Plinth", 4: "Interlock"}

COLOR_BY_TYPE = {0: "green", 1: "gray", 3: "lightblue", 4: "orange"}
NAME_BY_TYPE  = {0: "ground", 1: "column", 3: "office", 4: "core"}
SIZE_BY_TYPE  = {0: 20, 1: 8, 3: 12, 4: 14}

# --- re-run prediction (same as 13B) ---
pyg_vis = PyG.ByCSVPath(
    path=str(CSV_DIR),
    level="graph",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical",
)
pyg_vis.LoadModel(str(MODEL_PATH))
pyg_vis.SetHyperparameters(split=(0.0, 0.0, 1.0), shuffle=False)

pred_results = pyg_vis.Predict()
true_label   = int(pred_results["y_true"].tolist()[0])
pred_label   = int(pred_results["pred"].tolist()[0])
correct      = (true_label == pred_label)

print(f"True label : {true_label} — {MAPPING[true_label]}")
print(f"Predicted  : {pred_label} — {MAPPING[pred_label]}")
print(f"Correct    : {correct}")

# --- load graph from CSV (pattern from S06-15 section 16) ---
graphs = Graph.ByCSVPath(path=str(CSV_DIR))
g = graphs[0]
verts = Graph.Vertices(g)

nodes_df      = pd.read_csv(CSV_DIR / "nodes.csv")
label_by_node = {int(r["node_id"]): int(r["label"]) for _, r in nodes_df.iterrows()}

for i, v in enumerate(verts):
    d         = Topology.Dictionary(v)
    cell_type = label_by_node.get(i, 0)
    cell_color = COLOR_BY_TYPE.get(cell_type, "white")
    cell_name  = NAME_BY_TYPE.get(cell_type, str(cell_type))
    size       = SIZE_BY_TYPE.get(cell_type, 10)

    true_color = cell_color
    pred_color = cell_color if correct else "red"  # whole graph wrong → all nodes red

    d = Dictionary.SetValuesAtKeys(d,
        ["true_color", "pred_color", "size", "cell_name"],
        [true_color,   pred_color,   size,   cell_name])
    v = Topology.SetDictionary(v, d)

g = Graph.Reshape(g)

True label : 0 — Separation
Predicted  : 1 — Separation with Plinth
Correct    : False


In [19]:
# View 1 — True label (your assessment)
print(f"TRUE: {true_label} — {MAPPING[true_label]}")
Topology.Show(
    g,
    vertexSize=6,
    vertexSizeKey="size",
    vertexColorKey="true_color",
    showVertexLabel=True,
    vertexLabelKey="cell_name",
    backgroundColor="white",
    camera=[0, 0, 3],
    renderer=renderer
)

TRUE: 0 — Separation


In [20]:
# View 2 — Predicted label (model's assessment)
# nodes are red if the model disagreed with your label
print(f"PREDICTED: {pred_label} — {MAPPING[pred_label]}")
Topology.Show(
    g,
    vertexSize=6,
    vertexSizeKey="size",
    vertexColorKey="pred_color",
    showVertexLabel=True,
    vertexLabelKey="cell_name",
    backgroundColor="white",
    camera=[0, 0, 3],
    renderer=renderer
)

PREDICTED: 1 — Separation with Plinth


In [21]:
import os
out_path = os.path.join(os.path.dirname(SUPPORT_DIR), "BGR_graph.html")

Graph.PyvisGraph(
    graph,
    path=out_path,
    vertexSizeKey="size",
    vertexColorKey="color",
    vertexLabelKey="cell_name",
    edgeWeightKey="width",
    edgeColorKey="color"
)
print(f"Saved → {out_path}")

C:\Users\etmaglari\IAAC\etmaglari_gML\Homework03\BGR_graph.html
Saved → C:\Users\etmaglari\IAAC\etmaglari_gML\Homework03\BGR_graph.html
